# Práctica guiada · Cuatro problemas, una arquitectura

**Consigna:** tu equipo recibe **cuatro problemas**, uno por modalidad: imagen, secuencia, texto y tabular. Para cada uno tenés que decidir **qué arquitectura usarías, contra qué línea base, con qué métrica y con qué riesgo**, y si una red neuronal **se justifica**.

**Esta actividad es de pensar, no de codear.** Casi no hay código: lo único que escribís son palabras entre comillas. El código solo te devuelve **preguntas** para que revises tu decisión; nunca te da la respuesta. Los problemas son ficticios y no hay datos que procesar: lo que se evalúa es la decisión.

| Minutos | Qué hacen |
| --- | --- |
| 0–8 | Leer los cuatro problemas y completar un primer mapeo. **A los 8 minutos, control obligatorio.** |
| 8–13 | Completar línea base, métrica, riesgo y justificación. |
| 13–15 | Revisar las pistas e **intercambiar** con otro equipo. |

Tu número de equipo es el de tu tarjeta. Lo único que cambiás es el número en la sección 1 y los textos de la sección 3.

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook; si ya está todo, no hace nada.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "matplotlib": "matplotlib", "jinja2": "jinja2"}
INSTALAR_XGBOOST = False          # opcional: ponelo en True para sumar XGBoost a la comparación de modelos
if INSTALAR_XGBOOST: REQUISITOS["xgboost"] = "xgboost"

print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")


## 0 · Preparación (no hace falta tocarla)

Esta celda carga los cuatro problemas de cada equipo y las funciones que muestran la información y revisan tus decisiones.

In [ ]:
MODALIDADES = ['imagen', 'secuencia', 'texto', 'tabular']
FAMILIAS = ['red convolucional', 'red recurrente', 'transformer', 'regresión', 'árboles / boosting', 'modelo generativo', 'agrupamiento o similitud (sin etiquetas)']
METRICAS = ['exactitud', 'AUC', 'precisión', 'recall', 'F1', 'error medio (MAE)', 'precisión en los N de mayor riesgo', 'acierto en las N primeras sugerencias', 'costo total', 'revisión humana de una muestra']
RIESGOS = ['fuga', 'desbalance (eventos raros)', 'pocos datos etiquetados', 'sesgo', 'respuestas inventadas', 'cambio en el tiempo', 'costo de operar', 'falta de explicabilidad']
SINO = ['sí', 'no', 'depende']

PROBLEMAS = {'imagen': {1: ('Productos dañados en las fotos de una devolución',
                'Aprobar o rechazar la devolución según el daño.',
                '3.000 fotos etiquetadas; 1 de cada 8 muestra daño.',
                'Aprobar un producto dañado cuesta más que revisarlo a mano.',
                {'etiquetas': True, 'raro': False, 'orden': False, 'asimetrico': True, 'generar': False}),
            2: ('Clasificar las fotos del catálogo por categoría',
                'Asignar una categoría a cada producto nuevo.',
                '40.000 fotos, cada una con su categoría (12 categorías).',
                'Una categoría equivocada baja las ventas, pero se corrige fácil.',
                {'etiquetas': True, 'raro': False, 'orden': False, 'asimetrico': False, 'generar': False}),
            3: ('Estado de una góndola a partir de una foto',
                'Avisar al personal si la góndola está vacía o desordenada.',
                '800 fotos del personal, con estado: ok, vacía o desordenada.',
                'No avisar pierde ventas; avisar de más cansa al personal.',
                {'etiquetas': True, 'raro': False, 'orden': False, 'asimetrico': True, 'generar': False}),
            4: ('Detectar fotos duplicadas en el catálogo',
                'Eliminar las publicaciones repetidas.',
                '25.000 fotos; nadie marcó cuáles están repetidas.',
                'Eliminar una foto distinta por error borra un producto.',
                {'etiquetas': False, 'raro': False, 'orden': False, 'asimetrico': True, 'generar': False}),
            5: ('Verificar que la foto coincide con la descripción',
                'Bloquear publicaciones donde la foto no corresponde.',
                '10.000 publicaciones con foto y texto; 300 marcadas por reclamos.',
                'Bloquear una publicación correcta molesta al vendedor.',
                {'etiquetas': True, 'raro': True, 'orden': False, 'asimetrico': False, 'generar': False}),
            6: ('Contar productos en una foto de depósito',
                'Estimar el stock sin conteo manual.',
                '500 fotos con el conteo manual (de 5 a 60 productos).',
                'Un conteo con error distorsiona la reposición.',
                {'etiquetas': True, 'raro': False, 'orden': False, 'asimetrico': False, 'generar': False}),
            7: ('Leer etiquetas de precio en fotos',
                'Detectar etiquetas con un precio distinto al del sistema.',
                '2.000 fotos de etiquetas con el precio anotado en una planilla.',
                'Un precio erróneo expone a reclamos y multas.',
                {'etiquetas': True, 'raro': False, 'orden': False, 'asimetrico': True, 'generar': False})},
 'secuencia': {1: ('Ventas de la próxima semana por producto',
                   'Decidir cuánto reponer de cada producto.',
                   '3 años de ventas semanales de 400 productos.',
                   'Faltante pierde ventas; sobrante inmoviliza stock.',
                   {'etiquetas': True, 'raro': False, 'orden': True, 'asimetrico': False, 'generar': False}),
               2: ('Próxima compra de un cliente según su historial',
                   'Elegir qué producto ofrecerle.',
                   'Historial de compras de 8.000 clientes (unas 9 por año).',
                   'Una oferta equivocada se ignora: el costo es bajo.',
                   {'etiquetas': True, 'raro': False, 'orden': True, 'asimetrico': False, 'generar': False}),
               3: ('Anomalías en el consumo diario de una tienda',
                   'Avisar al gerente cuando un día es raro.',
                   '2 años de ventas diarias de 30 tiendas; nadie marcó anomalías.',
                   'Demasiadas alarmas falsas hacen que se ignoren.',
                   {'etiquetas': False, 'raro': False, 'orden': True, 'asimetrico': False, 'generar': False}),
               4: ('Cuándo un cliente deja de comprar según sus visitas',
                   'Contactar al cliente antes de que se vaya.',
                   'Visitas a la app y compras de 8.000 clientes; abandona el 14 %.',
                   'Contactar a quien no se iba cuesta 5; perder al cliente, 60.',
                   {'etiquetas': True, 'raro': False, 'orden': True, 'asimetrico': True, 'generar': False}),
               5: ('Tiempo de entrega según los eventos logísticos',
                   'Prometerle al cliente un plazo realista.',
                   'Eventos de 50.000 envíos (despacho, escala, intento) y la fecha final.',
                   'Prometer de menos genera reclamos; de más, pierde ventas.',
                   {'etiquetas': True, 'raro': False, 'orden': True, 'asimetrico': False, 'generar': False}),
               6: ('Fraude en una secuencia de transacciones',
                   'Frenar una operación sospechosa.',
                   '2 millones de transacciones; 0,1 % fue fraude.',
                   'Dejar pasar un fraude cuesta mucho más que frenar una operación válida.',
                   {'etiquetas': True, 'raro': True, 'orden': True, 'asimetrico': True, 'generar': False}),
               7: ('Demanda horaria del call center',
                   'Decidir cuántas personas poner en cada franja.',
                   '18 meses de llamadas por hora.',
                   'Poco personal alarga la espera; mucho es dinero sobrando.',
                   {'etiquetas': True, 'raro': False, 'orden': True, 'asimetrico': False, 'generar': False})},
 'texto': {1: ('Clasificar los reclamos por motivo',
               'Derivar cada reclamo al área correcta.',
               '6.000 reclamos escritos, con su motivo (5 motivos).',
               'Derivar mal demora la respuesta al cliente.',
               {'etiquetas': True, 'raro': False, 'orden': False, 'asimetrico': False, 'generar': False}),
           2: ('Detectar el tono de las reseñas',
               'Priorizar las reseñas muy negativas.',
               '20.000 reseñas con estrellas, pero sin etiqueta de tono.',
               'No ver una reseña muy negativa daña la reputación.',
               {'etiquetas': False, 'raro': False, 'orden': False, 'asimetrico': True, 'generar': False}),
           3: ('Extraer producto y problema de un mail',
               'Completar solo el formulario de reclamo.',
               '1.500 mails con el producto y el problema marcados a mano.',
               'Un dato mal extraído abre un reclamo equivocado.',
               {'etiquetas': True, 'raro': False, 'orden': False, 'asimetrico': False, 'generar': False}),
           4: ('Responder preguntas frecuentes',
               'Responder solo las consultas simples.',
               '400 preguntas frecuentes, cada una con su respuesta oficial.',
               'Una respuesta inventada genera un reclamo mayor.',
               {'etiquetas': True, 'raro': False, 'orden': False, 'asimetrico': True, 'generar': True}),
           5: ('Resumir hilos largos de atención',
               'Que un agente entienda un caso en 30 segundos.',
               '3.000 hilos de conversación; ninguno tiene resumen escrito.',
               'Un resumen que omite un dato clave provoca errores.',
               {'etiquetas': False, 'raro': False, 'orden': False, 'asimetrico': True, 'generar': True}),
           6: ('Detectar datos personales en comentarios',
               'Ocultar datos personales antes de publicar.',
               '5.000 comentarios; 4 % tiene datos personales marcados.',
               'Publicar un dato personal es grave; ocultar de más es leve.',
               {'etiquetas': True, 'raro': True, 'orden': False, 'asimetrico': True, 'generar': False}),
           7: ('Normalizar descripciones de productos',
               'Unificar las descripciones para el buscador.',
               '30.000 descripciones con variantes de escritura, sin etiquetas.',
               'Bajo: se puede revisar después.',
               {'etiquetas': False, 'raro': False, 'orden': False, 'asimetrico': False, 'generar': False})},
 'tabular': {1: ('Predecir el abandono de clientes',
                 'Elegir a quién contactar.',
                 '8.000 clientes, 21 variables; abandona el 14,3 %.',
                 'Contactar de más cuesta 5; perder a un cliente, 60.',
                 {'etiquetas': True, 'raro': False, 'orden': False, 'asimetrico': True, 'generar': False}),
             2: ('Estimar el valor de vida del cliente',
                 'Decidir cuánto invertir en retener a cada cliente.',
                 '8.000 clientes con su gasto de los siguientes 12 meses.',
                 'Sobrestimar el valor lleva a gastar de más.',
                 {'etiquetas': True, 'raro': False, 'orden': False, 'asimetrico': False, 'generar': False}),
             3: ('Predecir las devoluciones',
                 'Ofrecer ayuda antes de la compra.',
                 '11.000 pedidos; 8 % fue devuelto.',
                 'Ofrecer ayuda es barato; una devolución cuesta el envío dos veces.',
                 {'etiquetas': True, 'raro': False, 'orden': False, 'asimetrico': True, 'generar': False}),
             4: ('Segmentar clientes por comportamiento',
                 'Armar campañas distintas por grupo.',
                 '8.000 clientes; nadie definió grupos.',
                 'Grupos que no se pueden explicar no sirven.',
                 {'etiquetas': False, 'raro': False, 'orden': False, 'asimetrico': False, 'generar': False}),
             5: ('Probabilidad de pago a tiempo',
                 'Aprobar o limitar un crédito en cuotas.',
                 '5.000 créditos con su historial de pago; 9 % cayó en mora.',
                 'Aprobar a quien no paga cuesta mucho más que rechazar a quien sí pagaba.',
                 {'etiquetas': True, 'raro': False, 'orden': False, 'asimetrico': True, 'generar': False}),
             6: ('Predecir que un producto se quede sin stock',
                 'Adelantar un pedido a proveedores.',
                 'Stock diario de 400 productos durante 2 años; 3 % de los días con faltante.',
                 'El faltante pierde ventas; adelantar de más inmoviliza stock.',
                 {'etiquetas': True, 'raro': False, 'orden': True, 'asimetrico': True, 'generar': False}),
             7: ('Detectar operaciones sospechosas',
                 'Mandar una operación a revisión manual.',
                 '100.000 operaciones; 0,5 % confirmada como sospechosa por auditoría.',
                 'La revisión manual tiene un cupo diario: no se puede revisar todo.',
                 {'etiquetas': True, 'raro': True, 'orden': False, 'asimetrico': True, 'generar': False})}}

import re
import pandas as pd
CAMPOS = ["arquitectura", "linea_base", "metrica", "riesgo", "red_se_justifica", "por_que"]
SUPERVISADAS = ["red convolucional", "red recurrente", "transformer", "regresión", "árboles / boosting"]

def problema_de(equipo, modalidad):
    titulo, decision, datos, costo, rasgos = PROBLEMAS[modalidad][equipo]
    return dict(titulo=titulo, decision=decision, datos=datos, costo=costo, **rasgos)

def mostrar_problemas(equipo):
    print(f"EQUIPO {equipo} · tus cuatro problemas\n" + "=" * 60)
    for modalidad in MODALIDADES:
        p = problema_de(equipo, modalidad)
        print(f"\n[{modalidad.upper()}] {p['titulo']}\n  Se quiere decidir : {p['decision']}\n  Datos disponibles : {p['datos']}\n  Si se equivoca    : {p['costo']}")

def mostrar_listas():
    for nombre, lista in [("FAMILIAS", FAMILIAS), ("METRICAS", METRICAS), ("RIESGOS", RIESGOS), ("SINO", SINO)]:
        print(f"{nombre}:\n  " + "\n  ".join(f"- {x}" for x in lista) + "\n")

def pistas_problema(modalidad, p, r):
    """Devuelve preguntas, nunca la respuesta. Lista vacía = sin pistas."""
    vacios = [c for c in CAMPOS if not str(r.get(c, "")).strip()]
    if vacios: return ["Falta completar: " + ", ".join(vacios) + "."]
    h = []
    for campo, lista, nombre in [("arquitectura", FAMILIAS, "FAMILIAS"), ("metrica", METRICAS, "METRICAS"), ("riesgo", RIESGOS, "RIESGOS"), ("red_se_justifica", SINO, "SINO")]:
        if r[campo] not in lista: h.append(f"«{campo}» tiene que ser un valor de {nombre}: copialo tal cual de la lista.")
    if h: return h
    arq, met, rie = r["arquitectura"], r["metrica"], r["riesgo"]
    if arq == "red convolucional" and modalidad != "imagen": h.append("La convolución busca formas en imágenes: ¿tu dato es una imagen?")
    if arq == "red recurrente" and not p["orden"]: h.append("Una red recurrente lee datos con orden: ¿el orden importa en este problema?")
    if arq == "transformer" and modalidad == "tabular": h.append("Mirá el mapa de arquitecturas: en una tabla, ¿qué probarías antes de un transformer?")
    if arq == "modelo generativo" and not p["generar"]: h.append("Un modelo generativo produce contenido: ¿tu problema pide generar algo o decidir entre opciones?")
    if arq == "modelo generativo" and rie != "respuestas inventadas": h.append("Un modelo generativo suena seguro aunque se equivoque: ¿ese es tu riesgo principal?")
    if (not p["etiquetas"]) and arq in SUPERVISADAS and not re.search(r"preentrenad|sin etiquetas|aproxim|estrella|reglas|similitud", r["linea_base"] + " " + r["por_que"], re.I):
        h.append("En este problema nadie marcó la respuesta: ¿cómo entrenarías un modelo supervisado?")
    if met == "exactitud" and (p["raro"] or p["asimetrico"]): h.append("Acertar siempre «no» da una exactitud altísima y los errores no cuestan lo mismo: ¿qué métrica mira lo que importa?")
    if p["orden"] and rie not in ("fuga", "cambio en el tiempo", "desbalance (eventos raros)"): h.append("Este problema tiene orden en el tiempo: ¿qué riesgo trae mezclar el pasado con el futuro?")
    if r["red_se_justifica"] == "sí" and modalidad == "tabular": h.append("En las dos comparaciones de hoy, ¿cuándo ganó una red en una tabla?")
    if len(r["linea_base"].strip()) < 12: h.append("Escribí qué modelo o regla simple probarías primero.")
    if len(r["por_que"].strip()) < 40: h.append("Justificá en una o dos frases: ¿por qué esa arquitectura y no la más simple?")
    return h

def control_8(mapeo):
    print("CONTROL A LOS 8 MINUTOS: mostrale esto al facilitador\n" + "-" * 60)
    for modalidad in MODALIDADES:
        arq = str(mapeo.get(modalidad, {}).get("arquitectura", "")).strip()
        print(("✔ " if arq else "… ") + f"{modalidad:<10} {arq or 'pendiente'}")

def revisar(mapeo, equipo):
    print(f"REVISIÓN · equipo {equipo}: pistas en forma de pregunta, no respuestas\n" + "=" * 60)
    total = 0
    for modalidad in MODALIDADES:
        p = problema_de(equipo, modalidad); h = pistas_problema(modalidad, p, mapeo.get(modalidad, {})); total += len(h)
        print(f"\n[{modalidad.upper()}] {p['titulo']}")
        if h:
            for x in h: print("  → Pista:", x)
        else: print("  ✔ Sin pistas. Ahora tenés que poder defenderlo en voz alta.")
    print(f"\nTotal de pistas: {total}")
    return total

def resumen(mapeo, equipo):
    filas = []
    for modalidad in MODALIDADES:
        r = mapeo.get(modalidad, {}); p = problema_de(equipo, modalidad)
        filas.append({"modalidad": modalidad, "problema": p["titulo"], "arquitectura": r.get("arquitectura", ""), "línea base": r.get("linea_base", ""), "métrica": r.get("metrica", ""), "riesgo": r.get("riesgo", ""), "¿la red se justifica?": r.get("red_se_justifica", "")})
    pd.set_option("display.max_colwidth", 60, "display.width", 200)
    print(f"EQUIPO {equipo} · resumen para la puesta en común\n")
    print(pd.DataFrame(filas).set_index("modalidad").T.to_string())
    print("\nPor qué:")
    for modalidad in MODALIDADES: print(f"  [{modalidad}] {mapeo.get(modalidad, {}).get('por_que', '')}")

## 1 · Tu equipo

Cambiá el número por el de tu equipo (del 1 al 7) y corré la celda: vas a ver tus cuatro problemas.

In [ ]:
MI_EQUIPO = 1          # ← cambialo por el número de tu equipo

mostrar_problemas(MI_EQUIPO)

## 2 · Antes de elegir: preguntas para cada problema

Conversen antes de escribir. Para cada problema, respondan en voz alta:

1. **¿Qué tipo de dato es?** Fijate en el mapa de arquitecturas de la clase.
2. **¿Hay etiquetas, cuántas y hay orden en el tiempo?** Mirá la línea «Datos disponibles».
3. **¿Qué cuesta más: equivocarse para un lado o para el otro?** Mirá la línea «Si se equivoca».
4. **¿Qué probarías antes de una red?** Esa es tu línea base.

Estas son las listas de las que podés copiar valores en el próximo paso:

In [ ]:
mostrar_listas()

## 3 · Tu mapeo

Completá cada campo **entre comillas**. En `arquitectura`, `metrica`, `riesgo` y `red_se_justifica` copiá un valor tal cual de las listas de arriba; en `linea_base` y `por_que` escribí con tus palabras.

- `arquitectura`: la familia que elegirías.
- `linea_base`: el modelo o la regla simple que probarías primero.
- `metrica`: con qué medirías si funciona.
- `riesgo`: el riesgo principal.
- `red_se_justifica`: ¿una red mejora a la línea base lo suficiente para pagar su costo? (`sí`, `no` o `depende`).
- `por_que`: una o dos frases que justifiquen tu elección.

In [ ]:
mapeo = {
    "imagen":    {"arquitectura": "", "linea_base": "", "metrica": "", "riesgo": "", "red_se_justifica": "", "por_que": ""},
    "secuencia": {"arquitectura": "", "linea_base": "", "metrica": "", "riesgo": "", "red_se_justifica": "", "por_que": ""},
    "texto":     {"arquitectura": "", "linea_base": "", "metrica": "", "riesgo": "", "red_se_justifica": "", "por_que": ""},
    "tabular":   {"arquitectura": "", "linea_base": "", "metrica": "", "riesgo": "", "red_se_justifica": "", "por_que": ""},
}

## Control a los 8 minutos

Corré esta celda y mostrale el resultado al facilitador antes de seguir.

In [ ]:
control_8(mapeo)

## 4 · Revisión: pistas, no respuestas

**Predicción antes de correr:** ¿en cuántos de tus cuatro problemas esperás recibir alguna pista? Escribí un número del 0 al 4.

In [ ]:
prediccion = None      # ← escribí tu predicción, de 0 a 4

In [ ]:
total = revisar(mapeo, MI_EQUIPO)
if prediccion is not None: print(f"\nPredijiste {prediccion} y hubo {total}: ¿qué te sorprendió?")

Si hay pistas, **no corrijan de inmediato**: discutan primero qué supuesto del equipo cuestiona cada pregunta. Después ajusten el mapeo (sección 3) y vuelvan a correr la revisión.

## 5 · Para la puesta en común

In [ ]:
duda = ""      # ← ¿En qué problema dudaron más? ¿Qué dato, métrica o riesgo los hizo dudar?

resumen(mapeo, MI_EQUIPO)
print("\nDuda del equipo:", duda)

## 6 · Intercambio (últimos minutos)

Pasale tu resumen a otro equipo y pedile **una observación concreta**: ¿qué problema defendería distinto y por qué? Anotala acá:

In [ ]:
observacion_recibida = ""      # ← lo que te devolvió el otro equipo
print(observacion_recibida)